# Lesson 4: Auto-merging Retrieval

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import utils

import os

In [4]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI.pdf"]
).load_data()

In [5]:
print(type(documents), "\n")
print(len(documents), "\n")
print(type(documents[0]))
print(documents[0])

<class 'list'> 

41 

<class 'llama_index.core.schema.Document'>
Doc ID: 92278488-2374-4840-9597-a01868969fba
Text: PAGE 1 Founder, DeepLearning.AI Collected Insights from Andrew
Ng How to  Build Your Career in AI A Simple Guide


## Auto-merging retrieval setup

In [8]:
from llama_index.core import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

In [9]:
from llama_index.core.node_parser import HierarchicalNodeParser

# create the hierarchical node parser w/ default settings
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128]
)

In [10]:
nodes = node_parser.get_nodes_from_documents([document])

In [11]:
from llama_index.core.node_parser import get_leaf_nodes

leaf_nodes = get_leaf_nodes(nodes)
print(leaf_nodes[30].text)

Of course, I also encourage learning driven by curiosity. If something interests you, go ahead 
and learn it regardless of how useful it might turn out to be!  Maybe this will lead to a creative 
spark or technical breakthrough.
How much math do you need to know to be a machine learning engineer?


In [12]:
nodes_by_id = {node.node_id: node for node in nodes}

parent_node = nodes_by_id[leaf_nodes[30].parent_node.node_id]
print(parent_node.text)

On some days, maybe you’ll end up studying for an 
hour or longer.

PAGE 12
Should You 
Learn Math to 
Get a Job in AI? 
CHAPTER 3
LEARNING

PAGE 13
Should you Learn Math to Get a Job in AI? CHAPTER 3
Is math a foundational skill for AI? It’s always nice to know more math! But there’s so much to 
learn that, realistically, it’s necessary to prioritize. Here’s how you might go about strengthening 
your math background.
To figure out what’s important to know, I find it useful to ask what you need to know to make 
the decisions required for the work you want to do. At DeepLearning.AI, we frequently ask, 
“What does someone need to know to accomplish their goals?” The goal might be building a 
machine learning model, architecting a system, or passing a job interview.
Understanding the math behind algorithms you use is often helpful, since it enables you to 
debug them. But the depth of knowledge that’s useful changes over time. As machine learning 
techniques mature and become more reliabl

### Building the index

In [14]:
from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(
    model="gemma-4-31b-it",
    api_key=utils.get_google_api_key(),
    temperature=0.1,
)

2026-07-20 22:44:01,322 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"


In [15]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

2026-07-20 22:44:05,490 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:44:05,514 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-20 22:44:05,803 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:44:05,805 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-20 22:44:05,882 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-20 22:44:05,885 - INFO - Loading SentenceTransformer model from BAAI/b

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 22:44:08,268 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:08,555 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:08,841 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:09,128 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:09,421 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:44:09,443 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

In [16]:
from llama_index.core import VectorStoreIndex, StorageContext

storage_context = StorageContext.from_defaults()
storage_context.docstore.add_documents(nodes)

automerging_index = VectorStoreIndex(leaf_nodes, storage_context=storage_context)

automerging_index.storage_context.persist(persist_dir="./merging_index")

In [17]:
# This block of code is optional to check
# if an index file exist, then it will load it
# if not, it will rebuild it

import os
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage

if not os.path.exists("./merging_index"):
    storage_context = StorageContext.from_defaults()
    storage_context.docstore.add_documents(nodes)

    automerging_index = VectorStoreIndex(
            leaf_nodes,
            storage_context=storage_context,
        )

    automerging_index.storage_context.persist(persist_dir="./merging_index")
else:
    automerging_index = load_index_from_storage(
        StorageContext.from_defaults(persist_dir="./merging_index")
    )


2026-07-20 22:44:13,709 - INFO - Loading all indices.


### Defining the retriever and running the query engine

In [18]:
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

automerging_retriever = automerging_index.as_retriever(
    similarity_top_k=12
)

retriever = AutoMergingRetriever(
    automerging_retriever, 
    automerging_index.storage_context, 
    verbose=True
)

rerank = SentenceTransformerRerank(top_n=6, model="BAAI/bge-reranker-base")

auto_merging_engine = RetrieverQueryEngine.from_args(
    automerging_retriever, node_postprocessors=[rerank]
)

2026-07-20 22:44:14,037 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:14,039 - INFO - No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
2026-07-20 22:44:14,314 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-07-20 22:44:14,594 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:44:14,616 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-07-20 22:44:14,905 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:15,208 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-bas

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-20 22:44:15,884 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:16,177 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:16,489 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:16,903 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:44:17,205 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:44:17,239 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/tokenizer_config.json 

In [19]:
auto_merging_response = auto_merging_engine.query(
    "What is the importance of networking in AI?"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-20 22:44:22,096 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:44:35,025 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


In [21]:
from llama_index.core.response.notebook_utils import display_response

display_response(auto_merging_response)

**`Final Response:`** A strong professional network in AI is important because it can help propel you forward when you need advice or help. Beyond providing valuable information, people in your network can play an invaluable role by referring you to potential employers. Additionally, welcoming others into the AI community can help others recognize your expertise and encourage you to continue developing, while also reducing doubts that you are a member of the community.

## Putting it all Together

In [22]:
from utils import build_automerging_index, get_automerging_query_engine

In [23]:
from llama_index.llms.google_genai import GoogleGenAI

index = build_automerging_index(
    [document],
    llm=GoogleGenAI(
        model="gemini-2.5-flash",
        api_key=utils.get_google_api_key(),
        temperature=0.1,
    ),
    save_dir="./merging_index",
)


2026-07-20 22:52:36,702 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash "HTTP/1.1 200 OK"
2026-07-20 22:52:37,335 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:52:37,355 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-20 22:52:37,634 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:52:37,746 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-20 22:52:37,748 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-20 22:

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 22:52:40,123 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:40,422 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:40,742 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:41,015 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:41,284 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:52:41,305 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

In [24]:
query_engine = get_automerging_query_engine(index, similarity_top_k=6)

2026-07-20 22:52:44,212 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:44,215 - INFO - No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
2026-07-20 22:52:44,507 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-07-20 22:52:44,784 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:52:44,817 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-07-20 22:52:45,103 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:45,529 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-bas

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-20 22:52:45,947 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:46,228 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:46,554 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:46,831 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:47,167 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:52:47,188 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/tokenizer_config.json 

## Evaluation (LlamaIndex RAG triad)


In [25]:
from utils import get_eval_llm, evaluate_query_engine, summarize_eval_df

eval_llm = get_eval_llm()


2026-07-20 22:52:51,839 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"


### Two layers

In [26]:
auto_merging_index_0 = build_automerging_index(
    documents,
    llm=GoogleGenAI(
        model="gemma-4-31b-it",
        api_key=utils.get_google_api_key(),
        temperature=0.1,
    ),
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="merging_index_0",
    chunk_sizes=[2048,512],
)

2026-07-20 22:52:52,122 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-20 22:52:52,435 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:52:52,455 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-20 22:52:52,798 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:52:52,815 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-20 22:52:52,817 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-20 22:52

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 22:52:55,418 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:55,742 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:56,073 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:56,378 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:52:56,752 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:52:56,787 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

In [27]:
auto_merging_engine_0 = get_automerging_query_engine(
    auto_merging_index_0,
    similarity_top_k=12,
    rerank_top_n=6,
)

2026-07-20 22:53:00,976 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-20 22:53:00,979 - INFO - No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
2026-07-20 22:53:01,277 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-07-20 22:53:01,554 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:53:01,574 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-07-20 22:53:01,852 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:53:02,133 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-bas

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-20 22:53:02,516 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:53:02,786 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:53:03,141 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:53:03,418 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:53:03,696 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:53:03,716 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/tokenizer_config.json 

In [28]:
eval_questions = []
with open("generated_questions.text", "r") as file:
    for line in file:
        item = line.strip()
        if item:
            eval_questions.append(item)


In [ ]:
records_app0 = evaluate_query_engine(
    auto_merging_engine_0,
    eval_questions,
    app_id="app_0",
    llm=eval_llm,
)
records_app0.head()


2026-07-20 22:53:07,455 - INFO - > Merging 1 nodes into parent node.
> Parent node id: 31158589-a647-46bd-bacf-a0ffdaabb696.
> Parent node text: PAGE 26
If you’re considering a role switch, a startup can be an easier place to do it than a big...

2026-07-20 22:53:07,456 - INFO - > Merging 1 nodes into parent node.
> Parent node id: 427dcb81-2e10-424d-b5c2-c1972362e7c7.
> Parent node text: PAGE 25
Finding a job has a few predictable steps that include selecting the companies to which y...

2026-07-20 22:53:07,456 - INFO - > Merging 1 nodes into parent node.
> Parent node id: 678bfd4d-d03f-48e3-bc23-fb443f4b285d.
> Parent node text: PAGE 33
Choose who to work with. It’s tempting to take a position because of the projects you’ll ...

2026-07-20 22:53:07,457 - INFO - > Merging 1 nodes into parent node.
> Parent node id: 601f8e9c-a781-4370-9de7-93259f244f2f.
> Parent node text: PAGE 23
Each project is only one step on a longer journey, hopefully one that has a positive impa...

2026-07-20 2

> Merging 1 nodes into parent node.
> Parent node id: 31158589-a647-46bd-bacf-a0ffdaabb696.
> Parent node text: PAGE 26
If you’re considering a role switch, a startup can be an easier place to do it than a big...

> Merging 1 nodes into parent node.
> Parent node id: 427dcb81-2e10-424d-b5c2-c1972362e7c7.
> Parent node text: PAGE 25
Finding a job has a few predictable steps that include selecting the companies to which y...

> Merging 1 nodes into parent node.
> Parent node id: 678bfd4d-d03f-48e3-bc23-fb443f4b285d.
> Parent node text: PAGE 33
Choose who to work with. It’s tempting to take a position because of the projects you’ll ...

> Merging 1 nodes into parent node.
> Parent node id: 601f8e9c-a781-4370-9de7-93259f244f2f.
> Parent node text: PAGE 23
Each project is only one step on a longer journey, hopefully one that has a positive impa...

> Merging 1 nodes into parent node.
> Parent node id: 17af32d0-072b-4789-a1f8-85c56906cc21.
> Parent node text: PAGE 29
If you’re preparing to s

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-20 22:53:07,912 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:53:21,985 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-20 22:53:22,005 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:53:35,119 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:54:02,597 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:54:16,368 - INFO - > Merging 1 nodes into parent node.
> Parent node id: 037d6910-75d4-4bca-be46-5c7e1e313ad5.
> Parent node text: PAGE 27
There’s a lot we don’t know about the future: When will we cure Alzheimer’s disease? Who ...

2026-07-20 22:54:16,369 - INFO - > Merging 2 nodes into parent node.
> Parent node id: 26d9d53a-9bf4-4293-bc6f-cb8dc9a7e993.
> Parent node text: PAGE 20
Working on projects requires making tough choices about what to build and how to go 
abou...

2026-07-20 22:54:16,369 - INFO - > Merging 1 node

> Merging 1 nodes into parent node.
> Parent node id: 037d6910-75d4-4bca-be46-5c7e1e313ad5.
> Parent node text: PAGE 27
There’s a lot we don’t know about the future: When will we cure Alzheimer’s disease? Who ...

> Merging 2 nodes into parent node.
> Parent node id: 26d9d53a-9bf4-4293-bc6f-cb8dc9a7e993.
> Parent node text: PAGE 20
Working on projects requires making tough choices about what to build and how to go 
abou...

> Merging 1 nodes into parent node.
> Parent node id: 678bfd4d-d03f-48e3-bc23-fb443f4b285d.
> Parent node text: PAGE 33
Choose who to work with. It’s tempting to take a position because of the projects you’ll ...

> Merging 1 nodes into parent node.
> Parent node id: f5dbc2a3-7a7c-462b-837a-b4d742d416f2.
> Parent node text: PAGE 19
Develop a side hustle. Even if you have a full-time job, a fun project that may or may no...

> Merging 1 nodes into parent node.
> Parent node id: d88d0cbf-5bf1-4030-b783-74c54af2fd16.
> Parent node text: PAGE 18
It goes without saying t

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-20 22:54:17,111 - ERROR - Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x1076c1b40> is already entered
2026-07-20 22:54:17,114 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:54:34,301 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-20 22:54:34,308 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:54:49,421 - ERROR - Task was destroyed but it is pending!
task: <Task pending name='Task-125' coro=<Kernel.shell_main() running at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]>
2026-07-20 22:54:4

> Merging 2 nodes into parent node.
> Parent node id: 26d9d53a-9bf4-4293-bc6f-cb8dc9a7e993.
> Parent node text: PAGE 20
Working on projects requires making tough choices about what to build and how to go 
abou...

> Merging 1 nodes into parent node.
> Parent node id: af4e8832-396d-4287-a2af-ccdd1182332a.
> Parent node text: PAGE 7
These phases apply in a wide 
range of professions, but AI 
involves unique elements.
For ...

> Merging 1 nodes into parent node.
> Parent node id: 371fe0e6-e45f-47da-adc9-b327766f62e7.
> Parent node text: PAGE 15
One of the most important skills of an AI architect is the ability to identify ideas that...

> Merging 1 nodes into parent node.
> Parent node id: 7e7d6679-526c-4ec1-95b7-1c432303a17c.
> Parent node text: PAGE 22
Over the course of a career, you’re likely to work on projects in succession, each growin...

> Merging 1 nodes into parent node.
> Parent node id: d88d0cbf-5bf1-4030-b783-74c54af2fd16.
> Parent node text: PAGE 18
It goes without saying t

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-20 22:55:29,047 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:55:30,612 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 503 Service Unavailable"
2026-07-20 22:55:30,615 - WARNING - Retrying llama_index.llms.google_genai.base.GoogleGenAI._chat in 1 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.
2026-07-20 22:55:31,622 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:56:00,132 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-20 22:56:00,152 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:56:15,726 - INFO - AFC is enabled with max remote calls: 10.


In [ ]:
summarize_eval_df(records_app0)


In [ ]:
print("Auto-merging app_0 evaluation complete.")


In [ ]:
pass  # reserved


In [ ]:
pass  # reserved


### Three layers

In [ ]:
auto_merging_index_1 = build_automerging_index(
    documents,
    llm=GoogleGenAI(
        model="gemini-2.5-flash",
        api_key=utils.get_google_api_key(),
        temperature=0.1,
    ),
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="merging_index_1",
    chunk_sizes=[2048,512,128],
)

In [ ]:
auto_merging_engine_1 = get_automerging_query_engine(
    auto_merging_index_1,
    similarity_top_k=12,
    rerank_top_n=6,
)


In [ ]:
records_app1 = evaluate_query_engine(
    auto_merging_engine_1,
    eval_questions,
    app_id="app_1",
    llm=eval_llm,
)
records_app1.head()


In [ ]:
summarize_eval_df(records_app1)


In [ ]:
import pandas as pd

comparison = pd.concat(
    [summarize_eval_df(records_app0), summarize_eval_df(records_app1)],
    ignore_index=True,
)
comparison


In [ ]:
print("Auto-merging comparison complete.")
